# TCGer ArcFace student encoder (Colab, GPU)

Trains a fast card-identity embedding to A/B against the shipped DINOv2-small encoder.

**Recipe** — TCG-AR (arXiv 2607.02090): one class per catalog card, ArcFace (s=30, m=0.5),
synthetic augmentations only (~a handful of views per card per epoch). Student backbone:
**FastViT-T8** (ANE ~1 ms class) with a 384-d embedding head, so the exported model keeps the
shipped index format (count x 384, int8 scale 127).

**Contract (do not change):** input `image` 224x224 RGB, scale 1/255 with ImageNet mean/std
baked into the graph; the shortest-edge-256 -> center-crop-224 resize stays in Swift
(`CardEmbeddingEncoder.swift`), matching the shipped DINOv2 conversion. Output `embedding`,
L2-normalized, 384-d.

**Inputs on Drive**
- `MyDrive/TCGer-encoder/CardsIndexMetadata.json` — the repo's shipped metadata (defines annIndex order)
- card images: `MyDrive/UniFi Drive_UNAS Pro 8/.../card-library/pokemon/...` (same roots as the
  parity-v2 index notebook); rows missing on Drive are downloaded from each entry's `imageURL`

**Outputs to `MyDrive/TCGer-encoder/`**
- `arcface-checkpoint.pt` (resumable, per epoch)
- `CardEmbeddings-arcface.mlpackage` (zipped)
- `CardsIndexVectors-arcface.bin` + `arcface-eval.json` (recall metrics vs the current encoder)


In [ ]:
# 1) Config, Drive, GPU
from google.colab import drive
drive.mount('/content/drive')

import os, torch
assert torch.cuda.is_available(), "Switch runtime to a GPU (L4/T4)."
print(torch.cuda.get_device_name(0))

ENC_DIR   = "/content/drive/MyDrive/TCGer-encoder"
META_PATH = f"{ENC_DIR}/CardsIndexMetadata.json"
OLD_BIN   = f"{ENC_DIR}/CardsIndexVectors-dinov2-current.bin"   # optional, for A/B eval
LIB_CANDIDATES = [
    "/content/drive/MyDrive/UniFi Drive_UNAS Pro 8/UNAS Pro 8_Main Backup/Images/pvc-19daba96-3902-4005-aab6-60b80b8f171a/card-library/pokemon",
]
CACHE_DIR = "/content/card-images"          # local copy; Drive streaming is too slow to train from
CKPT      = f"{ENC_DIR}/arcface-checkpoint.pt"

EMBED_DIM   = 384
BACKBONE    = "fastvit_t8.apple_in1k"        # timm; ANE-friendly
EPOCHS      = 12
VIEWS_PER_CARD_PER_EPOCH = 3
BATCH       = 256
LR          = 3e-4
ARC_S, ARC_M = 30.0, 0.50
IMG_SIZE    = 224
SEED        = 22

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(ENC_DIR, exist_ok=True)
assert os.path.exists(META_PATH), "Copy the repo's CardsIndexMetadata.json to MyDrive/TCGer-encoder/ first."


In [ ]:
# 2) Dependencies
%pip -q install timm==1.0.* coremltools==8.* pillow


In [ ]:
# 3) Catalog rows + image resolution (same filename layouts as the parity-v2 index notebook),
#    then one-time local caching. Missing rows download from the entry's imageURL.
import json, shutil, urllib.request, concurrent.futures as cf
from pathlib import Path

entries = json.load(open(META_PATH))
entries.sort(key=lambda e: e["annIndex"])
for i, e in enumerate(entries):
    assert e["annIndex"] == i, "annIndex order must be contiguous"
print("catalog rows:", len(entries))

lib_root = next((Path(p) for p in LIB_CANDIDATES if Path(p).exists()), None)
print("drive library:", lib_root)

def drive_candidates(card_id: str):
    if lib_root is None: return []
    out = []
    stem = card_id
    out += [lib_root/"images"/f"{stem}.webp", lib_root/"images"/f"{stem}.png", lib_root/"images"/f"{stem}.jpg"]
    if "-" in card_id:
        set_code, num = card_id.split("-", 1)
        out += [lib_root/"images"/set_code/num/"high.webp", lib_root/"images"/set_code/f"{num}.webp",
                lib_root/"images"/set_code/f"{card_id}.webp"]
    return out

def cached_path(i: int) -> Path:
    return Path(CACHE_DIR)/f"{i:05d}.img"

def materialize(i_entry):
    i, e = i_entry
    dst = cached_path(i)
    if dst.exists() and dst.stat().st_size > 0: return None
    for cand in drive_candidates(e["cardId"]):
        if cand.exists():
            shutil.copyfile(cand, dst); return None
    url = e.get("imageURL")
    if not url: return i
    try:
        with urllib.request.urlopen(url, timeout=30) as r, open(dst, "wb") as f:
            shutil.copyfileobj(r, f)
        return None
    except Exception:
        return i

with cf.ThreadPoolExecutor(16) as ex:
    missing = [m for m in ex.map(materialize, enumerate(entries)) if m is not None]
print("unresolvable images:", len(missing))
assert len(missing) < len(entries) * 0.02, f"too many missing images: {len(missing)}"
VALID = [i for i in range(len(entries)) if i not in set(missing)]


In [ ]:
# 4) Dataset: TCG-AR style synthetic views. Each __getitem__ returns one randomly
#    augmented view of one catalog card; the label is the card's annIndex.
import random, io
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader

IMNET_MEAN = [0.485, 0.456, 0.406]; IMNET_STD = [0.229, 0.224, 0.225]

def contract_resize(img: Image.Image) -> Image.Image:
    # Mirrors CardEmbeddingEncoder.swift: scale so shortest edge >= 256 (and both sides
    # cover 224), bicubic with ceil, then center-crop 224.
    import math
    w, h = img.size
    s = max(256/min(w, h), IMG_SIZE/w, IMG_SIZE/h)
    rw, rh = math.ceil(w*s), math.ceil(h*s)
    img = img.resize((rw, rh), Image.BICUBIC)
    l, t = (rw-IMG_SIZE)//2, (rh-IMG_SIZE)//2
    return img.crop((l, t, l+IMG_SIZE, t+IMG_SIZE))

class CardViews(Dataset):
    def __init__(self, indices, train=True):
        self.indices = indices; self.train = train
    def __len__(self):
        return len(self.indices) * (VIEWS_PER_CARD_PER_EPOCH if self.train else 1)
    def __getitem__(self, k):
        i = self.indices[k % len(self.indices)]
        img = Image.open(cached_path(i)).convert("RGB")
        if self.train:
            if random.random() < 0.85:
                img = T.RandomPerspective(distortion_scale=0.35, p=1.0,
                                          fill=random.randint(0, 255))(img)
            if random.random() < 0.8:
                img = ImageEnhance.Brightness(img).enhance(random.uniform(0.55, 1.45))
                img = ImageEnhance.Color(img).enhance(random.uniform(0.6, 1.4))
                img = ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.3))
            if random.random() < 0.5:
                img = img.filter(ImageFilter.GaussianBlur(random.uniform(0.5, 2.2)))
            elif random.random() < 0.3:
                img = ImageEnhance.Sharpness(img).enhance(random.uniform(1.2, 2.5))
        x = TF.to_tensor(contract_resize(img))
        if self.train and random.random() < 0.5:
            x = x + torch.randn_like(x) * random.uniform(0.005, 0.03)
            x = x.clamp(0, 1)
        return TF.normalize(x, IMNET_MEAN, IMNET_STD), i

train_loader = DataLoader(CardViews(VALID, train=True), batch_size=BATCH, shuffle=True,
                          num_workers=8, pin_memory=True, drop_last=True,
                          persistent_workers=True)
print("train views/epoch:", len(train_loader.dataset))


In [ ]:
# 5) Model + ArcFace head + resumable training
import math, time, timm
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(BACKBONE, pretrained=True, num_classes=0)
        feat = self.backbone.num_features
        self.proj = nn.Linear(feat, EMBED_DIM)
    def forward(self, x):
        return F.normalize(self.proj(self.backbone(x)), dim=-1)

class ArcFace(nn.Module):
    def __init__(self, classes):
        super().__init__()
        self.w = nn.Parameter(torch.empty(classes, EMBED_DIM))
        nn.init.xavier_uniform_(self.w)
    def forward(self, emb, labels):
        cos = emb @ F.normalize(self.w, dim=-1).t()
        theta = torch.acos(cos.clamp(-1+1e-7, 1-1e-7))
        target = torch.cos(theta + ARC_M)
        onehot = F.one_hot(labels, self.w.shape[0]).to(cos.dtype)
        return ARC_S * (onehot * target + (1-onehot) * cos)

device = "cuda"
model, head = Encoder().to(device), ArcFace(len(entries)).to(device)
opt = torch.optim.AdamW([*model.parameters(), *head.parameters()], lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler()
start_epoch = 0

if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=device)
    model.load_state_dict(ck["model"]); head.load_state_dict(ck["head"])
    opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
    start_epoch = ck["epoch"] + 1
    print(f"resumed after epoch {ck['epoch']}")

for epoch in range(start_epoch, EPOCHS):
    model.train(); head.train()
    t0, seen, loss_sum, correct = time.time(), 0, 0.0, 0
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            emb = model(x)
            logits = head(emb, y)
            loss = F.cross_entropy(logits, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        seen += y.numel(); loss_sum += loss.item() * y.numel()
        correct += (logits.argmax(1) == y).sum().item()
    sched.step()
    torch.save({"model": model.state_dict(), "head": head.state_dict(),
                "opt": opt.state_dict(), "sched": sched.state_dict(),
                "epoch": epoch, "config": {"backbone": BACKBONE, "dim": EMBED_DIM}}, CKPT)
    print(f"epoch {epoch}: loss {loss_sum/seen:.3f}  train-acc {correct/seen:.3f}  {(time.time()-t0)/60:.1f} min")


In [ ]:
# 6) Eval: catalog self-retrieval. Gallery = clean contract-resized cards; queries =
#    fresh augmented views. Reports recall@1/@5, and the same metric for the shipped
#    DINOv2 int8 index if CardsIndexVectors-dinov2-current.bin is present.
@torch.no_grad()
def embed_all(ds, bs=512):
    model.eval()
    out = torch.empty(len(ds), EMBED_DIM)
    loader = DataLoader(ds, batch_size=bs, num_workers=8)
    row = 0
    for x, _ in loader:
        e = model(x.to(device))
        out[row:row+len(e)] = e.cpu(); row += len(e)
    return out

gallery = embed_all(CardViews(VALID, train=False))
queries = embed_all(CardViews(VALID, train=True))          # 3 augmented views per card
qlabels = torch.tensor([VALID[k % len(VALID)] for k in range(len(CardViews(VALID, train=True)))])
glabels = torch.tensor(VALID)

def recall(q, ql, g, gl, ks=(1, 5)):
    sims = q @ g.t()
    top = sims.topk(max(ks), dim=1).indices
    hits = gl[top] == ql[:, None]
    return {f"recall@{k}": hits[:, :k].any(1).float().mean().item() for k in ks}

student = recall(queries, qlabels, gallery, glabels)
print("student:", student)

baseline = None
if os.path.exists(OLD_BIN):
    import struct
    raw = open(OLD_BIN, "rb").read()
    n, d = struct.unpack("<ii", raw[:8])
    old = torch.tensor(np.frombuffer(raw, dtype=np.int8, offset=8).reshape(n, d).astype(np.float32) / 127.0)
    old = F.normalize(old, dim=-1)
    print("(baseline recall needs DINOv2 query embeddings — measured on-device via the replay harness instead; "
          "gallery loaded OK:", old.shape, ")")
json.dump({"student": student, "epochs": EPOCHS, "backbone": BACKBONE},
          open(f"{ENC_DIR}/arcface-eval.json", "w"), indent=1)


In [ ]:
# 7) Export: Core ML mlpackage (contract identical to the shipped encoder) + int8 index bin
import coremltools as ct
import struct

class Deploy(nn.Module):
    # Bakes /255 + ImageNet normalize into the graph, mirroring convert-dinov2-coreml.py:
    # the CoreML input is 0-255 pixels scaled by 1/255 via ImageType.
    def __init__(self, m):
        super().__init__()
        self.m = m
        self.register_buffer("mean", torch.tensor(IMNET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(IMNET_STD).view(1, 3, 1, 1))
    def forward(self, x):
        return self.m((x - self.mean) / self.std)

model_cpu = Deploy(model.float().cpu()).eval()
example = torch.rand(1, 3, IMG_SIZE, IMG_SIZE)
traced = torch.jit.trace(model_cpu, example)
ml = ct.convert(
    traced,
    convert_to="mlprogram",
    minimum_deployment_target=ct.target.iOS18,
    compute_units=ct.ComputeUnit.ALL,
    inputs=[ct.ImageType(name="image", shape=example.shape, scale=1/255.0, color_layout=ct.colorlayout.RGB)],
    outputs=[ct.TensorType(name="embedding")],
)
ml.save("/content/CardEmbeddings-arcface.mlpackage")
!cd /content && zip -qr "$ENC_DIR/CardEmbeddings-arcface.mlpackage.zip" CardEmbeddings-arcface.mlpackage
print("saved mlpackage zip")

# Index: embed EVERY catalog row (missing images keep zero vectors, matching absent rows)
full_gallery = torch.zeros(len(entries), EMBED_DIM)
full_gallery[torch.tensor(VALID)] = gallery
q = torch.clamp(torch.round(full_gallery * 127), -127, 127).to(torch.int8).numpy()
with open(f"{ENC_DIR}/CardsIndexVectors-arcface.bin", "wb") as f:
    f.write(struct.pack("<ii", len(entries), EMBED_DIM))
    f.write(q.tobytes())
print("saved index bin", len(entries), "x", EMBED_DIM)


## Local A/B test (back on the Mac)

1. Make a **worktree** so main stays untouched, then drop in the artifacts:
   - unzip `CardEmbeddings-arcface.mlpackage.zip` over `TCGer/Resources/ScanIndex/CardEmbeddings.mlpackage`
   - copy `CardsIndexVectors-arcface.bin` over `TCGer/Resources/ScanIndex/CardsIndexVectors.bin`
   - **delete `TCGer/Resources/ScanIndex/CardFaceGate.json`** — the gate was trained on DINOv2
     embeddings; with it absent the strategy runs ungated (its policy handles a nil gate).
2. Build-for-testing and run, against the same corpus as always:
   - `ScannerPerfOptionsTests` + `ScannerFixtureTests` (fixture floors)
   - `DevModeSessionReplayTests` via `-testPlan TCGer-Replay -test-timeouts-enabled NO` with
     `TEST_RUNNER_DEVMODE_SESSIONS_DIR` at the Session-Reference `sessions/` dir
3. Compare the DEVREPLAY summary + per-frame diffs against the shipped encoder's baseline
   (31/76 labeled correct as of 2026-08-21). Wrong accepts are disqualifying; watch the known
   foil clusters (ex13-18 Absol, dp1-125 energy, me05 foils) for the hoped-for accuracy gains.
4. If retrieval quality holds, retrain the rejection gate before shipping
   (`backend/src/scripts/train-rejection-gate.ts` — needs a crop dataset; see the handoff doc).
